# 노트북 13. RAG + GraphRAG

> Phase 4 — 지식 확장: 외부 문서를 활용하는 법

모델이 학습하지 않은 내부 문서를 기반으로 답변하게 하려면 RAG가 필요합니다.
그리고 단순 RAG로 답할 수 없는 복잡한 질문에는 GraphRAG가 대안이 됩니다.

**학습 목표**
- RAG(Retrieve → Augment → Generate) 파이프라인을 이해하고 구현할 수 있다
- 문서 로딩, 청크 분할, 벡터 저장, 검색의 전체 흐름을 구성할 수 있다
- LCEL 체인으로 RAG를 구현하고, 검색 품질을 개선할 수 있다
- GraphRAG의 개념을 이해하고, NetworkX로 간단한 지식 그래프를 구축할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | RAG 파이프라인 + GraphRAG 개념 | 읽고 실행 |
| Part 2 — 실습 | Chroma RAG + FAISS 전환 + 지식 그래프 | 직접 작성 |
| Part 3 — 챌린지 | 실제 문서 기반 RAG Q&A 시스템 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langchain langchain-text-splitters chromadb faiss-cpu networkx matplotlib

import os
import json
import numpy as np
from google import genai

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "text-embedding-004"

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

embedding_func = GoogleGenerativeAIEmbeddings(
    model=f"models/{EMBEDDING_MODEL}",
    google_api_key=GEMINI_API_KEY,
)

print("환경 설정 완료")

---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 RAG란?

**RAG**(Retrieval-Augmented Generation, 검색 증강 생성)는
LLM이 학습하지 않은 외부 문서를 검색하여 답변에 활용하는 기법입니다.

```
사용자 질문: "우리 회사 연차 정책은?"
    ↓
[Retrieve] 질문을 임베딩 → 벡터 스토어에서 유사 문서 검색
    ↓
[Augment]  검색된 문서를 프롬프트에 주입
    ↓
[Generate] LLM이 문서 기반으로 답변 생성
```

### 왜 RAG가 필요한가?

| 방법 | 장점 | 단점 |
|------|------|------|
| Fine-tuning | 모델 자체에 지식 내장 | 비용 높음, 데이터 업데이트 어려움 |
| Long Context | 전체 문서를 프롬프트에 넣기 | 토큰 비용 폭발, 길어지면 정확도 하락 |
| **RAG** | 필요한 부분만 검색하여 주입 | 검색 품질에 의존 |

> RAG는 비용 효율적이면서도 최신 정보를 반영할 수 있는 가장 실용적인 방법입니다.
> 문서가 업데이트되면 벡터 스토어만 갱신하면 됩니다.

## 1.2 RAG 파이프라인 전체 흐름

RAG 파이프라인은 크게 **인덱싱**(준비)과 **질의**(실행) 두 단계로 나뉩니다.

```
[인덱싱 — 1회 수행]
문서 로딩 → 청크 분할 → 임베딩 → 벡터 스토어 저장

[질의 — 매 질문마다]
질문 임베딩 → 유사 청크 검색 → 프롬프트 조합 → LLM 답변
```

이번 노트북에서는 이 전체 흐름을 처음부터 끝까지 구현합니다.

## 1.3 문서 준비: 샘플 데이터

실습용으로 가상의 회사 정책 문서를 만들어 사용합니다.
실제 서비스에서는 PDF, 웹 페이지, 데이터베이스 등 다양한 소스에서 문서를 로딩합니다.

In [ ]:
# 가상의 회사 정책 문서 (샘플 데이터)
company_docs = [
    """[연차 정책]
입사 1년 미만: 월 1일 발생 (최대 11일)
입사 1년 이상: 연 15일 발생
입사 3년 이상: 매 2년마다 1일 추가 (최대 25일)
연차는 1년 내 미사용 시 소멸되며, 연차 수당으로 전환할 수 없습니다.
반차(0.5일) 사용이 가능하며, 연속 5일 이상 사용 시 2주 전 신청이 필요합니다.""",

    """[원격근무 정책]
전 직원 주 2회 원격근무가 가능합니다.
원격근무일에는 오전 10시까지 메신저로 출근을 알려야 합니다.
팀 회의가 있는 날은 사무실 출근이 원칙입니다.
해외 원격근무는 최대 1개월까지 가능하며, 팀장 승인이 필요합니다.
원격근무 장비(노트북, 모니터)는 회사에서 지원합니다.""",

    """[복리후생]
점심 식대: 1일 12,000원 지원 (법인카드)
통신비: 월 50,000원 지원
도서 구입비: 분기당 100,000원 한도
건강검진: 연 1회 종합검진 (배우자 포함)
자기개발비: 연 1,000,000원 (교육, 컨퍼런스, 자격증)
경조사 지원: 본인 결혼 200만원, 가족 경조사 10~50만원""",

    """[평가 및 보상]
성과 평가는 반기 1회(6월, 12월) 실시합니다.
평가 등급: S(최우수), A(우수), B(보통), C(개선필요)
연봉 인상률: S등급 10%, A등급 7%, B등급 4%, C등급 0%
성과급: 연 1회(2월), 기본급의 0~200% (등급별 차등)
승진 심사: 연 1회(1월), 직급별 최소 근속 연수 충족 필요""",

    """[교육 및 성장]
신입사원 온보딩: 2주간 OJT 프로그램 운영
사내 스터디: 분기별 모집, 활동비 팀당 50만원 지원
외부 컨퍼런스: 연 2회까지 참가비 전액 지원 (국내/해외)
사내 강의: 전문 분야 강의 진행 시 건당 20만원 강사료
멘토링 프로그램: 입사 6개월 이내 직원 대상, 시니어 멘토 배정""",

    """[보안 정책]
사내 자료는 승인 없이 외부 반출할 수 없습니다.
USB 저장장치 사용은 보안팀 승인 후에만 가능합니다.
개인 기기로 사내 메일 접속 시 MDM 설치가 필요합니다.
퇴사 시 모든 사내 데이터를 반납하고, 개인 기기에서 삭제해야 합니다.
보안 교육은 분기 1회 필수이며, 미이수 시 시스템 접근이 제한됩니다.""",
]

print(f"문서 {len(company_docs)}개 준비 완료")
for i, doc in enumerate(company_docs):
    title = doc.split('\n')[0]
    print(f"  [{i}] {title} ({len(doc)}자)")

## 1.4 청크 분할

문서가 길면 전체를 하나의 임베딩으로 만들기 어렵습니다.
**청크 분할**(Chunking)은 문서를 적절한 크기로 나누는 과정입니다.

| 파라미터 | 역할 | 권장값 |
|----------|------|--------|
| `chunk_size` | 청크 최대 문자 수 | 200~1000 (도메인에 따라) |
| `chunk_overlap` | 인접 청크 간 겹치는 문자 수 | chunk_size의 10~20% |

> **chunk_overlap**이 필요한 이유: 청크 경계에서 문맥이 끊기는 것을 방지합니다.
> 겹치는 부분이 있으면, 한 청크에서 놓친 정보를 다음 청크에서 포착할 수 있습니다.

아래 코드는 `RecursiveCharacterTextSplitter`로 문서를 분할합니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Document 객체 생성
documents = []
for i, text in enumerate(company_docs):
    title = text.split('\n')[0].strip('[]')
    documents.append(Document(
        page_content=text,
        metadata={"source": f"policy_{i}", "title": title},
    ))

# 청크 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       # 청크 최대 200자
    chunk_overlap=30,     # 30자 겹침
    separators=["\n\n", "\n", ". ", " "],  # 분할 우선순위
)

chunks = text_splitter.split_documents(documents)

print(f"원본 문서: {len(documents)}개")
print(f"분할 청크: {len(chunks)}개\n")

# 처음 3개 청크 확인
for i, chunk in enumerate(chunks[:3]):
    print(f"--- 청크 {i} ({len(chunk.page_content)}자) ---")
    print(f"메타데이터: {chunk.metadata}")
    print(f"{chunk.page_content[:100]}...\n")

In [ ]:
# chunk_size에 따른 분할 결과 비교sample_text = company_docs[0]  # 연차 정책for size in [100, 200, 500]:    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=20)    result = splitter.split_text(sample_text)    print(f"chunk_size={size}: {len(result)}개 청크")    for i, chunk in enumerate(result):        print(f"  [{i}] ({len(chunk)}자) {chunk[:50]}...")    print()

> **RecursiveCharacterTextSplitter**는 `separators` 순서대로 분할을 시도합니다.
> 먼저 `\n\n`(빈 줄)으로 나누고, 그래도 `chunk_size`를 초과하면 `\n`, `. ` 순으로 분할합니다.
> 이렇게 하면 단락, 문장 단위의 자연스러운 분할이 가능합니다.

## 1.5 벡터 스토어에 저장

분할된 청크를 임베딩하여 Chroma 벡터 스토어에 저장합니다.
이것이 RAG 인덱싱 단계의 마지막입니다.

### 문서 로더 (Document Loaders)실제 서비스에서는 다양한 형식의 문서를 로딩해야 합니다.LangChain은 다양한 로더를 제공합니다.| 로더 | 형식 | 사용법 ||------|------|--------|| `TextLoader` | .txt | `TextLoader("file.txt")` || `PyPDFLoader` | .pdf | `PyPDFLoader("file.pdf")` || `CSVLoader` | .csv | `CSVLoader("file.csv")` || `WebBaseLoader` | 웹페이지 | `WebBaseLoader("https://...")` || `JSONLoader` | .json | `JSONLoader("file.json")` |```pythonfrom langchain_community.document_loaders import PyPDFLoaderloader = PyPDFLoader("manual.pdf")pages = loader.load()  # List[Document], 페이지별 분리```> 이 노트북에서는 문자열 데이터를 직접 사용하지만,> 프로덕션에서는 위 로더들로 실제 파일을 로딩합니다.

In [ ]:
from langchain_community.vectorstores import Chroma

# Chroma에 청크 저장
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_func,
    collection_name="company_policy",
)

print(f"벡터 스토어 저장 완료: {len(chunks)}개 청크")

# Retriever 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# 검색 테스트
test_docs = retriever.invoke("연차 며칠이에요?")
print(f"\n검색 결과: {len(test_docs)}개")
for doc in test_docs:
    print(f"  [{doc.metadata['title']}] {doc.page_content[:60]}...")

## 1.6 LCEL RAG 체인

LangChain Expression Language(LCEL)로 RAG 파이프라인을 하나의 체인으로 구성합니다.

```python
chain = retriever | prompt | model | parser
```

| 단계 | 역할 | 입력 → 출력 |
|------|------|------------|
| retriever | 유사 문서 검색 | 질문 → Document 리스트 |
| prompt | 프롬프트 조합 | context + question → 프롬프트 |
| model | LLM 호출 | 프롬프트 → AIMessage |
| parser | 텍스트 추출 | AIMessage → 문자열 |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# RAG 프롬프트 템플릿
rag_prompt = ChatPromptTemplate.from_template(
    """아래 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요.

[참고 문서]
{context}

[질문]
{question}"""
)

# 검색 결과를 텍스트로 변환하는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL RAG 체인
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")
print("구조: retriever → format_docs → prompt → llm → parser")

In [ ]:
# RAG 체인 실행
question = "입사 2년차 직원의 연차는 며칠인가요?"
answer = rag_chain.invoke(question)

print(f"Q: {question}")
print(f"A: {answer}")

### RAG 프롬프트 설계 팁| 팁 | 예시 ||-----|------|| 범위 제한 | "아래 문서만 참고하여 답하세요" || 모르면 인정 | "문서에 없는 내용은 '알 수 없습니다'라고 답하세요" || 출처 요구 | "어떤 문서를 참고했는지 함께 답하세요" || 형식 지정 | "3문장 이내로 요약하여 답하세요" || 인용 | "관련 부분을 직접 인용하세요" |> 가장 중요한 것은 **"문서에 없으면 모른다고 답하라"**입니다.> 이 지시 없이는 LLM이 학습 데이터를 기반으로 답변을 지어낼 수 있습니다 (할루시네이션).

In [ ]:
# 여러 질문으로 RAG 테스트
questions = [
    "원격근무는 몇 번 가능한가요?",
    "점심 식대는 얼마인가요?",
    "S등급 연봉 인상률은?",
    "USB 사용하려면 어떻게 해야 하나요?",
    "회사 주가는 얼마인가요?",  # 문서에 없는 질문
]

for q in questions:
    a = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {a}\n")

> 마지막 질문("회사 주가")은 문서에 없는 내용입니다.
> 프롬프트에 "문서에 없는 내용은..."이라는 지시를 넣었기 때문에,
> 모델이 없는 정보를 지어내는 **할루시네이션**을 방지할 수 있습니다.

## 1.7 검색 결과 확인 (디버깅)

RAG의 품질은 검색 품질에 크게 의존합니다.
답변이 잘못되었을 때, 검색된 문서를 확인하는 것이 디버깅의 첫 단계입니다.

In [ ]:
# 검색 결과를 함께 반환하는 RAG 체인
from langchain_core.runnables import RunnableParallel

rag_chain_with_source = RunnableParallel(
    answer=rag_chain,
    source_docs=retriever,
)

result = rag_chain_with_source.invoke("자기개발비는 얼마까지 지원되나요?")

print(f"답변: {result['answer']}\n")
print("=== 참조 문서 ===")
for i, doc in enumerate(result['source_docs']):
    print(f"  [{i}] [{doc.metadata['title']}] {doc.page_content[:80]}...")

In [ ]:
# 할루시네이션 감지: 검색 결과와 답변 대조# 답변이 검색된 문서에 근거하는지 확인하는 간단한 패턴def check_grounding(question, answer, source_docs):    """답변이 문서에 근거하는지 LLM으로 검증합니다."""    sources_text = "\n".join(d.page_content for d in source_docs)    check_prompt = ChatPromptTemplate.from_template(        """질문에 대한 답변이 아래 문서에 근거하는지 판단하세요.[문서]{sources}[질문] {question}[답변] {answer}근거 여부를 "근거함" 또는 "근거 없음"으로 답하고, 이유를 한 줄로 설명하세요."""    )    chain = check_prompt | llm | StrOutputParser()    return chain.invoke({"sources": sources_text, "question": question, "answer": answer})# 테스트q = "점심 식대는 얼마인가요?"result = rag_chain_with_source.invoke(q)grounding = check_grounding(q, result['answer'], result['source_docs'])print(f"Q: {q}")print(f"A: {result['answer']}")print(f"검증: {grounding}")

> `RunnableParallel`을 사용하면 답변과 참조 문서를 동시에 반환할 수 있습니다.
> 이 패턴은 프로덕션에서 "출처 표시" 기능을 구현할 때 유용합니다.

## 1.8 검색 품질 개선

RAG 성능을 높이는 핵심 전략들:

| 전략 | 설명 | 효과 |
|------|------|------|
| **k 값 조정** | 검색 문서 수 조절 | 너무 적으면 누락, 너무 많으면 노이즈 |
| **MMR** | 다양성 고려 검색 | 중복 문서 방지 |
| **청크 크기** | 문서 분할 단위 조절 | 너무 작으면 문맥 손실, 너무 크면 노이즈 |
| **메타데이터 필터** | 카테고리별 필터링 | 관련 없는 문서 제거 |
| **리랭킹** | 검색 결과를 LLM으로 재정렬 | 정확도 향상, 비용 증가 |

In [ ]:
# k 값에 따른 검색 품질 비교
query = "신입사원 교육은 어떻게 진행되나요?"

for k in [1, 3, 5]:
    docs = vectorstore.similarity_search(query, k=k)
    titles = [d.metadata['title'] for d in docs]
    print(f"k={k}: {titles}")

print("\nk가 클수록 더 많은 문서를 검색하지만, 관련 없는 문서도 포함될 수 있습니다")

In [ ]:
# MMR vs Similarity 비교
query = "회사에서 지원해주는 것들"

# 일반 유사도 검색
sim_docs = vectorstore.similarity_search(query, k=4)
print("=== Similarity ===")
for doc in sim_docs:
    print(f"  [{doc.metadata['title']}] {doc.page_content[:50]}...")

# MMR 검색
mmr_docs = vectorstore.max_marginal_relevance_search(query, k=4, fetch_k=10)
print("\n=== MMR ===")
for doc in mmr_docs:
    print(f"  [{doc.metadata['title']}] {doc.page_content[:50]}...")

print("\nMMR은 다양한 정책 문서에서 골고루 검색합니다")

### RAG 평가의 3요소 (RAG Triad)노트북 14에서 자세히 다루지만, RAG 품질을 판단하는 핵심 지표를 미리 소개합니다.| 지표 | 질문 | 비유 ||------|------|------|| **Context Relevance** | 검색된 문서가 질문과 관련 있는가? | 올바른 책을 찾았는가? || **Answer Faithfulness** | 답변이 검색된 문서에 근거하는가? | 책에 있는 내용만 말했는가? || **Answer Relevance** | 답변이 원래 질문에 대한 답인가? | 질문에 맞는 답을 했는가? |> 세 지표 모두 높아야 좋은 RAG입니다.> 하나라도 낮으면: 검색 품질(Retrieve) 또는 생성 품질(Generate)을 개선해야 합니다.

## 1.9 스트리밍 RAG

RAG 체인도 스트리밍이 가능합니다.
`chain.stream()`을 사용하면 답변이 실시간으로 생성됩니다.

In [ ]:
# 스트리밍 RAG
print("Q: 원격근무 규칙을 자세히 알려주세요\n")
print("A: ", end="")

for chunk in rag_chain.stream("원격근무 규칙을 자세히 알려주세요"):
    print(chunk, end="", flush=True)

print()

## 1.10 FAISS로 전환

노트북 12에서 배운 것처럼, LangChain의 통일된 인터페이스 덕분에
벡터 스토어를 Chroma에서 FAISS로 쉽게 전환할 수 있습니다.

In [ ]:
from langchain_community.vectorstores import FAISS

# 동일한 청크를 FAISS에 저장
faiss_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_func,
)

# FAISS Retriever로 RAG 체인 재구성
faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": 3})

rag_chain_faiss = (
    {"context": faiss_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 동일 질문으로 비교
question = "성과급은 언제 지급되나요?"
answer_faiss = rag_chain_faiss.invoke(question)

print(f"Q: {question}")
print(f"A (FAISS): {answer_faiss}")
print("\nRetriever만 교체하면 나머지 체인은 동일하게 동작합니다")

In [ ]:
# Multi-Query Retriever: 같은 질문을 여러 관점에서 검색# 하나의 질문으로 여러 검색 쿼리를 생성하여 검색 범위를 넓힙니다from langchain.retrievers.multi_query import MultiQueryRetrievermulti_retriever = MultiQueryRetriever.from_llm(    retriever=retriever,    llm=llm,)# 하나의 질문 → 여러 관점의 검색results = multi_retriever.invoke("회사에서 자기개발을 위해 지원해주는 것들은?")print(f"Multi-Query 검색 결과: {len(results)}개 문서")for doc in results:    print(f"  [{doc.metadata['title']}] {doc.page_content[:50]}...")print("\n내부적으로 질문을 여러 방식으로 재구성하여 검색합니다")

---

## 1.11 GraphRAG 개요

일반 RAG는 **단일 청크 검색**에 의존합니다.
하지만 다음과 같은 질문에는 한계가 있습니다:

| 질문 유형 | 일반 RAG의 한계 |
|-----------|----------------|
| **멀티홉**: "김 대리의 팀장은 누구이고, 그 팀장의 프로젝트는?" | 두 청크를 연결해야 함 |
| **전체 요약**: "이 문서의 핵심 주제 3가지는?" | 하나의 청크에 전체 맥락 없음 |
| **관계 추론**: "마케팅팀과 개발팀이 협업한 프로젝트는?" | 관계 정보가 청크에 흩어져 있음 |

**GraphRAG**는 이 문제를 **지식 그래프**(Knowledge Graph)로 해결합니다.

```
1. 문서에서 엔티티(사람, 조직, 개념)와 관계를 추출
2. 지식 그래프 구축 (노드 + 엣지)
3. 질문과 관련된 서브그래프를 탐색
4. 서브그래프 정보를 context로 LLM에 전달
```

## 1.12 지식 그래프 구축: 엔티티/관계 추출

LLM을 사용하여 텍스트에서 엔티티와 관계를 추출합니다.
노트북 8에서 배운 Structured Output을 활용합니다.

In [ ]:
# 지식 그래프용 샘플 데이터 — 가상의 조직도
org_text = """프로젝트 알파는 개발팀이 주도하고, 마케팅팀이 지원합니다.
김철수는 개발팀 팀장이며, 프로젝트 알파의 총괄 책임자입니다.
이영희는 마케팅팀 팀장이며, 프로젝트 알파의 마케팅 전략을 담당합니다.
박민수는 개발팀 소속으로, 프로젝트 알파의 백엔드 개발을 맡고 있습니다.
최지은은 마케팅팀 소속으로, 프로젝트 알파의 SNS 홍보를 담당합니다.
프로젝트 베타는 데이터팀이 주도하며, 개발팀이 기술 지원합니다.
정하늘은 데이터팀 팀장이며, 프로젝트 베타의 총괄 책임자입니다.
김철수와 정하늘은 프로젝트 베타에서 협업하고 있습니다."""

print(org_text)

### RAG + 대화 기록 (Conversational RAG)멀티턴 대화에서 RAG를 사용하려면, 이전 대화 맥락을 고려한 검색이 필요합니다.```사용자: "연차 정책 알려줘"  → 검색: "연차 정책"사용자: "반차도 가능해?"    → 검색: "반차" + 이전 맥락(연차 정책)``````pythonfrom langchain.chains import create_history_aware_retriever# 대화 기록을 고려한 검색 쿼리 생성history_aware_retriever = create_history_aware_retriever(    llm, retriever, contextualize_prompt)```> 단순 RAG는 매 질문을 독립적으로 검색합니다.> 대화형 RAG는 이전 맥락을 반영하여 검색 쿼리를 재구성합니다.

In [ ]:
# LLM으로 엔티티와 관계 추출
extraction_prompt = ChatPromptTemplate.from_template(
    """아래 텍스트에서 엔티티(사람, 팀, 프로젝트)와 관계를 추출하세요.

JSON 형식으로 반환하세요:
{{
  "entities": [{{"name": "이름", "type": "person|team|project"}}],
  "relations": [{{"source": "주체", "relation": "관계", "target": "대상"}}]
}}

텍스트:
{text}"""
)

extraction_chain = extraction_prompt | llm | StrOutputParser()
raw_result = extraction_chain.invoke({"text": org_text})

# JSON 파싱 — 코드 블록 제거
clean = raw_result.strip()
if clean.startswith("```"):
    clean = clean.split("\n", 1)[1]
if clean.endswith("```"):
    clean = clean.rsplit("```", 1)[0]

graph_data = json.loads(clean)

print(f"엔티티: {len(graph_data['entities'])}개")
for e in graph_data['entities']:
    print(f"  [{e['type']}] {e['name']}")

print(f"\n관계: {len(graph_data['relations'])}개")
for r in graph_data['relations']:
    print(f"  {r['source']} --[{r['relation']}]--> {r['target']}")

## 1.13 NetworkX로 지식 그래프 구축

추출된 엔티티와 관계를 **NetworkX** 그래프에 저장하고 시각화합니다.
NetworkX는 순수 Python 그래프 라이브러리로, 설치가 간편하고 교육/프로토타입에 적합합니다.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 지식 그래프 생성
G = nx.DiGraph()  # 방향 그래프

# 엔티티를 노드로 추가
for entity in graph_data['entities']:
    G.add_node(entity['name'], type=entity['type'])

# 관계를 엣지로 추가
for rel in graph_data['relations']:
    G.add_edge(rel['source'], rel['target'], relation=rel['relation'])

print(f"노드: {G.number_of_nodes()}개")
print(f"엣지: {G.number_of_edges()}개")

# 시각화
plt.figure(figsize=(12, 8))

# 노드 타입별 색상
color_map = {"person": "#4ECDC4", "team": "#FF6B6B", "project": "#45B7D1"}
node_colors = [color_map.get(G.nodes[n].get('type', ''), '#gray') for n in G.nodes]

pos = nx.spring_layout(G, k=2, seed=42)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=9)
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20)

# 엣지 라벨 (관계명)
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, font_color='red')

plt.title("Knowledge Graph")
plt.legend(
    handles=[
        plt.scatter([], [], c=color_map['person'], s=100, label='Person'),
        plt.scatter([], [], c=color_map['team'], s=100, label='Team'),
        plt.scatter([], [], c=color_map['project'], s=100, label='Project'),
    ],
    loc='upper left',
)
plt.axis('off')
plt.tight_layout()
plt.show()

## 1.14 그래프 탐색으로 질의 응답

지식 그래프의 핵심은 **관계를 따라가며 정보를 탐색**하는 것입니다.
질문과 관련된 엔티티를 찾고, 그 주변 노드를 탐색하여 context를 구성합니다.

> **추출 품질 향상 팁**> - 추출 프롬프트에 **예시**(few-shot)를 포함하면 정확도가 높아집니다> - Structured Output(노트북 8)을 사용하면 JSON 파싱 오류를 줄일 수 있습니다> - 긴 문서는 청크별로 추출한 후 결과를 병합합니다> - 추출된 결과를 사람이 검수하는 **Human-in-the-Loop** 과정이 이상적입니다

In [ ]:
def graph_search(G, query_entity, depth=2):
    """엔티티를 중심으로 depth만큼 탐색하여 관련 정보를 수집합니다."""
    if query_entity not in G.nodes:
        return f"'{query_entity}'을(를) 그래프에서 찾을 수 없습니다."

    # BFS로 주변 노드 탐색
    visited = set()
    context_lines = []
    queue = [(query_entity, 0)]

    while queue:
        node, d = queue.pop(0)
        if node in visited or d > depth:
            continue
        visited.add(node)

        # 나가는 엣지
        for _, target, data in G.out_edges(node, data=True):
            context_lines.append(f"{node}은(는) {target}에 대해 '{data['relation']}' 관계입니다.")
            queue.append((target, d + 1))

        # 들어오는 엣지
        for source, _, data in G.in_edges(node, data=True):
            context_lines.append(f"{source}은(는) {node}에 대해 '{data['relation']}' 관계입니다.")
            queue.append((source, d + 1))

    return "\n".join(context_lines)

# 그래프 탐색 테스트
context = graph_search(G, "김철수", depth=2)
print("=== '김철수' 중심 탐색 결과 ===")
print(context)

In [ ]:
# 그래프 context를 LLM에 전달하여 답변 생성
graph_rag_prompt = ChatPromptTemplate.from_template(
    """아래 지식 그래프 정보를 참고하여 질문에 답하세요.

[지식 그래프]
{context}

[질문]
{question}"""
)

graph_rag_chain = graph_rag_prompt | llm | StrOutputParser()

# 멀티홉 질문 테스트
questions = [
    ("김철수", "김철수는 어떤 프로젝트에 참여하고 있나요?"),
    ("프로젝트 알파", "프로젝트 알파에 참여하는 사람들은 누구인가요?"),
    ("김철수", "김철수와 협업하는 다른 팀장은 누구이고, 그 사람이 맡은 프로젝트는 무엇인가요?"),
]

for entity, question in questions:
    context = graph_search(G, entity, depth=2)
    answer = graph_rag_chain.invoke({"context": context, "question": question})
    print(f"Q: {question}")
    print(f"A: {answer}\n")

> 마지막 질문은 **멀티홉** 질문입니다.
> "김철수 → 협업 → 정하늘 → 프로젝트 베타"로 2단계를 거쳐야 답할 수 있습니다.
> 일반 RAG에서는 이런 관계 추론이 어렵지만, GraphRAG는 그래프 탐색으로 해결합니다.

In [ ]:
# 그래프 통계 분석# 어떤 노드가 가장 중요한지(연결이 많은지) 확인print("=== 그래프 통계 ===")print(f"노드: {G.number_of_nodes()}, 엣지: {G.number_of_edges()}")# 차수 중심성 — 연결이 많은 노드 = 중요한 엔티티degree_centrality = nx.degree_centrality(G)print("\n연결 중심성 (높을수록 중요):")for node, score in sorted(degree_centrality.items(), key=lambda x: -x[1])[:5]:    node_type = G.nodes[node].get('type', 'unknown')    print(f"  {node} [{node_type}]: {score:.3f}")print("\n가장 연결이 많은 노드가 그래프의 핵심 엔티티입니다")

## 1.15 RAG vs GraphRAG 비교

| 비교 항목 | RAG | GraphRAG |
|-----------|-----|----------|
| 검색 방식 | 벡터 유사도 | 그래프 탐색 |
| 강점 | 단순 질문, 사실 확인 | 관계 추론, 멀티홉 |
| 약점 | 관계 추론 어려움 | 구축 비용 높음 |
| 구축 비용 | 낮음 (임베딩만) | 높음 (엔티티/관계 추출) |
| 업데이트 | 문서 추가/삭제 쉬움 | 그래프 재구축 필요 |
| 적합한 경우 | FAQ, 문서 검색 | 조직도, 관계 분석 |

> **실무 권장**: 대부분의 경우 RAG로 충분합니다.
> GraphRAG는 관계 기반 질문이 핵심인 도메인(조직 관리, 법률, 의학)에서 고려하세요.

> **프로덕션 그래프 저장소**: 이 노트북에서는 NetworkX(순수 Python)를 사용했지만,
> 대규모 그래프에서는 **Neo4j**가 사실상 표준입니다.
> Neo4j AuraDB(클라우드) + LangChain `Neo4jGraph` 통합으로 확장할 수 있습니다.
> NetworkX는 교육/프로토타입용이며, 영속성과 대규모 탐색에는 한계가 있습니다.

---

### 생각해보기

1. RAG에서 검색된 문서가 질문과 관련이 없으면 어떤 문제가 생길까요? 이를 감지하고 대응하는 방법은 무엇일까요?
2. chunk_size를 50으로 줄이면 검색 정확도가 높아질까요, 낮아질까요? 반대로 1000으로 늘리면 어떨까요?
3. GraphRAG에서 LLM이 엔티티/관계를 잘못 추출하면 어떤 문제가 생길까요? 이를 검증하는 방법은 무엇일까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### Local Search vs Global SearchGraphRAG의 두 가지 검색 전략:| 전략 | 방식 | 적합한 질문 ||------|------|------------|| **Local Search** | 특정 엔티티 중심 탐색 | "김철수의 역할은?" || **Global Search** | 전체 그래프 요약 | "이 조직의 프로젝트 현황은?" |이 노트북에서 구현한 `graph_search()`는 **Local Search**에 해당합니다.Global Search를 구현하려면:1. 그래프를 커뮤니티(클러스터)로 분할2. 각 커뮤니티의 요약을 LLM으로 생성3. 질문에 관련된 커뮤니티 요약을 context로 활용> Global Search는 Microsoft의 GraphRAG 논문에서 제안된 방법입니다.> 구현이 복잡하므로, 실제 필요할 때 `graphrag` 라이브러리를 참고하세요.

### 실습 13-1: Chroma 기반 RAG 파이프라인 구축

IT 기술 문서를 사용하여 RAG 파이프라인을 처음부터 구축하세요.

**요구사항**
1. `tech_docs` 리스트로 Document 객체 생성 + 청크 분할 (chunk_size=150, overlap=20)
2. Chroma 벡터 스토어에 저장
3. LCEL RAG 체인 구성 (retriever | prompt | llm | parser)
4. 3개 질문으로 테스트

In [ ]:
tech_docs = [
    """[Docker]
Docker는 애플리케이션을 컨테이너로 패키징하는 플랫폼입니다.
컨테이너는 애플리케이션과 모든 의존성을 포함하여 어디서든 동일하게 실행됩니다.
Dockerfile로 이미지를 빌드하고, docker run으로 컨테이너를 실행합니다.
Docker Compose는 여러 컨테이너를 하나의 설정 파일로 관리합니다.""",

    """[Kubernetes]
Kubernetes(K8s)는 컨테이너 오케스트레이션 플랫폼입니다.
여러 서버에 걸쳐 컨테이너를 자동으로 배포, 확장, 관리합니다.
Pod는 K8s의 최소 배포 단위이며, 하나 이상의 컨테이너를 포함합니다.
Service는 Pod에 대한 네트워크 접근을 제공하고, Ingress는 외부 트래픽을 라우팅합니다.""",

    """[CI/CD]
CI(Continuous Integration)는 코드 변경을 자동으로 빌드하고 테스트하는 프로세스입니다.
CD(Continuous Deployment)는 테스트를 통과한 코드를 자동으로 배포합니다.
GitHub Actions, Jenkins, GitLab CI 등이 대표적인 CI/CD 도구입니다.
파이프라인은 빌드 → 테스트 → 배포 단계로 구성됩니다.""",
]

# TODO: RAG 파이프라인 구축 + 테스트

# 정답
# docs = [
#     Document(page_content=text, metadata={"title": text.split('\n')[0].strip('[]')})
#     for text in tech_docs
# ]
#
# splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
# tech_chunks = splitter.split_documents(docs)
# print(f"청크: {len(tech_chunks)}개")
#
# tech_store = Chroma.from_documents(
#     documents=tech_chunks,
#     embedding=embedding_func,
#     collection_name="tech_docs",
# )
#
# tech_retriever = tech_store.as_retriever(search_kwargs={"k": 3})
#
# tech_rag = (
#     {"context": tech_retriever | format_docs, "question": RunnablePassthrough()}
#     | rag_prompt
#     | llm
#     | StrOutputParser()
# )
#
# for q in ["Docker와 Kubernetes의 차이점은?", "CI/CD 파이프라인은 어떤 단계로 구성되나요?", "Pod란 무엇인가요?"]:
#     a = tech_rag.invoke(q)
#     print(f"Q: {q}")
#     print(f"A: {a}\n")

print("TODO를 완성해주세요")

### 실습 13-2: FAISS로 전환하여 비교

실습 13-1과 동일한 데이터를 FAISS로 구축하고 결과를 비교하세요.

**요구사항**
1. 동일한 `tech_chunks`를 FAISS에 저장
2. FAISS 기반 RAG 체인 구성
3. 동일한 질문으로 Chroma 결과와 비교
4. 차이가 있다면 원인을 코드 주석으로 설명

### 실무 의사결정 가이드```질문: 외부 문서 기반 답변이 필요한가?  ├─ No → LLM 직접 호출 (노트북 1~6)  └─ Yes → 질문이 관계 추론을 포함하는가?       ├─ No → RAG (이 노트북 전반부)       └─ Yes → GraphRAG 검토 (이 노트북 후반부)              └─ 데이터 규모가 큰가?                   ├─ No → NetworkX (프로토타입)                   └─ Yes → Neo4j (프로덕션)```

In [ ]:
# TODO: FAISS RAG + Chroma 비교

# 정답
# # 실습 13-1의 tech_chunks를 사용합니다. 먼저 실습 13-1을 완성하세요.
# tech_faiss = FAISS.from_documents(
#     documents=tech_chunks,
#     embedding=embedding_func,
# )
#
# faiss_tech_retriever = tech_faiss.as_retriever(search_kwargs={"k": 3})
#
# tech_rag_faiss = (
#     {"context": faiss_tech_retriever | format_docs, "question": RunnablePassthrough()}
#     | rag_prompt
#     | llm
#     | StrOutputParser()
# )
#
# question = "Docker Compose는 무엇인가요?"
#
# answer_chroma = tech_rag.invoke(question)
# answer_faiss = tech_rag_faiss.invoke(question)
#
# print(f"Q: {question}")
# print(f"A (Chroma): {answer_chroma}")
# print(f"A (FAISS):  {answer_faiss}")
# # 검색된 문서가 같으면 답변도 유사합니다.
# # 차이가 나는 경우는 거리 함수(L2)와 검색 알고리즘의 미세한 차이 때문입니다.

print("TODO를 완성해주세요")

### 실습 13-3: NetworkX로 지식 그래프 구축 + 시각화

아래 텍스트에서 엔티티와 관계를 추출하고, 지식 그래프를 구축하세요.

**요구사항**
1. LLM으로 `school_text`에서 엔티티/관계를 JSON으로 추출
2. NetworkX 그래프에 노드/엣지 추가
3. 그래프를 시각화
4. 특정 엔티티 중심으로 탐색 + LLM 답변 생성

In [ ]:
school_text = """한국대학교 컴퓨터공학과에는 이 교수, 박 교수, 최 교수가 있습니다.
이 교수는 인공지능 연구실을 운영하며, 대학원생 김민준과 이서연을 지도합니다.
박 교수는 보안 연구실을 운영하며, 대학원생 정우진을 지도합니다.
최 교수는 데이터베이스 연구실을 운영합니다.
김민준은 자연어 처리를 연구하고 있으며, 최근 논문을 국제학회에 발표했습니다.
이서연은 컴퓨터 비전을 연구하고 있습니다.
이 교수와 박 교수는 AI 보안 공동 프로젝트를 진행하고 있습니다."""

# TODO: 엔티티/관계 추출 → 그래프 구축 → 시각화 → 질의

# 정답
# raw = extraction_chain.invoke({"text": school_text})
# clean = raw.strip()
# if clean.startswith("```"):
#     clean = clean.split("\n", 1)[1]
# if clean.endswith("```"):
#     clean = clean.rsplit("```", 1)[0]
# school_data = json.loads(clean)
#
# G2 = nx.DiGraph()
# for e in school_data['entities']:
#     G2.add_node(e['name'], type=e['type'])
# for r in school_data['relations']:
#     G2.add_edge(r['source'], r['target'], relation=r['relation'])
#
# print(f"노드: {G2.number_of_nodes()}, 엣지: {G2.number_of_edges()}")
#
# plt.figure(figsize=(12, 8))
# pos = nx.spring_layout(G2, k=2, seed=42)
# nx.draw_networkx_nodes(G2, pos, node_size=1200, alpha=0.8)
# nx.draw_networkx_labels(G2, pos, font_size=8)
# nx.draw_networkx_edges(G2, pos, arrows=True, arrowsize=15)
# edge_labels = nx.get_edge_attributes(G2, 'relation')
# nx.draw_networkx_edge_labels(G2, pos, edge_labels, font_size=7)
# plt.title("School Knowledge Graph")
# plt.axis('off')
# plt.tight_layout()
# plt.show()
#
# context = graph_search(G2, "이 교수", depth=2)
# answer = graph_rag_chain.invoke({
#     "context": context,
#     "question": "이 교수가 지도하는 학생들은 각각 무엇을 연구하나요?",
# })
# print(f"Q: 이 교수가 지도하는 학생들은 각각 무엇을 연구하나요?")
# print(f"A: {answer}")

print("TODO를 완성해주세요")

### 실습 13-4: 출처 포함 RAG 체인

답변과 함께 참조한 문서 출처를 반환하는 RAG 체인을 구성하세요.

**요구사항**
1. `RunnableParallel`로 답변 + 참조 문서를 동시에 반환
2. 참조 문서의 title 메타데이터를 출처로 표시
3. "건강검진 혜택은?"으로 테스트

### 청크 분할 전략 정리| 전략 | chunk_size | overlap | 적합한 경우 ||------|-----------|---------|------------|| 작은 청크 | 100~200 | 10~20 | 짧은 사실 확인 질문 || 중간 청크 | 300~500 | 30~50 | 일반적인 Q&A || 큰 청크 | 500~1000 | 50~100 | 문맥이 중요한 질문 |> 정답은 없습니다. 도메인과 질문 유형에 따라 실험하여 최적값을 찾아야 합니다.> **작게 시작해서 점점 키우는 것**이 일반적인 접근입니다.

In [ ]:
# TODO: 출처 포함 RAG

# 정답
# source_chain = RunnableParallel(
#     answer=rag_chain,
#     sources=retriever,
# )
#
# result = source_chain.invoke("건강검진 혜택은?")
#
# print(f"답변: {result['answer']}\n")
# print("출처:")
# seen = set()
# for doc in result['sources']:
#     title = doc.metadata['title']
#     if title not in seen:
#         seen.add(title)
#         print(f"  - {title}")

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 13-1에서 chunk_size를 50과 500으로 바꿔보면 검색 결과와 답변 품질이 어떻게 달라지나요?
2. 실습 13-3에서 LLM이 엔티티/관계를 누락하거나 잘못 추출한 경우가 있었나요? 이를 개선하려면 프롬프트를 어떻게 수정해야 할까요?
3. RAG와 GraphRAG를 결합하여 사용한다면, 어떤 질문에 어떤 방식을 적용할지 어떻게 결정하겠습니까?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 13-1: 실제 문서 기반 RAG Q&A 시스템 (난이도: 상)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

아래 조건을 만족하는 RAG Q&A 시스템을 설계하고 구현하세요.

**조건**
- 긴 텍스트 문서(1000자 이상)를 직접 작성하거나, 위키피디아에서 가져오기
- 적절한 chunk_size/overlap을 설계하고 그 이유를 설명
- RAG 체인 구성 + 출처 표시
- 최소 5개 질문으로 테스트 (문서에 있는 질문 3개 + 없는 질문 2개)

**추가 과제**
- k값(1, 3, 5)에 따른 답변 품질 비교
- similarity vs MMR 검색 결과 비교
- 답변에 할루시네이션이 있는지 검증

**힌트**
- 긴 텍스트를 준비하기 어려우면, `company_docs`를 확장하여 사용하세요
- 프롬프트에 "문서에 없는 내용은 모른다고 답하라"는 지시를 포함하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

---

### 생각해보기

1. RAG 시스템에서 문서가 주기적으로 업데이트된다면(예: 회사 정책 변경), 벡터 스토어를 어떻게 관리해야 할까요? 전체 재인덱싱 vs 증분 업데이트의 트레이드오프는?
2. 사용자가 같은 질문을 반복하는 경우, 매번 검색+LLM 호출을 하는 것은 비효율적입니다. 이를 최적화하는 방법은 무엇일까요?
3. RAG를 프로덕션에 배포한다면, 검색 품질을 지속적으로 모니터링하기 위해 어떤 지표를 추적해야 할까요? (힌트: 노트북 14에서 배울 평가 기법과 연결됩니다)